# PDF Combiner — All Iterations

For every `Results/_iteration_<N>/` folder, this notebook builds one combined
PDF per `(language, embedding model, ML model)` group containing the available
plots (Confusion Matrix → Per-Class Metrics → Precision-Recall → ROC), each
with `(a)`–`(d)` sub-captions.

- **Output**: a sibling folder `Results/_iteration_<N>_/` for each input
  iteration (note the trailing underscore). Files are named
  `<language>__<embedding>__<ml_model>.pdf`.
- Each output folder is wiped and rebuilt on every run.
- Plots are embedded as **vector** content (`Page.show_pdf_page`) and never
  upscaled, so quality is preserved for `\includegraphics` in LaTeX.
- Page width = A4; page height is trimmed to the actual content (no extra
  whitespace), capped at A4 height.

Iterations whose filenames don't follow the
`<plot>_<lang>_manual_<embedding>_<ml_model>` pattern (e.g. iteration_0 /
iteration_1) will produce an empty `_iteration_<N>_` folder.

Requires **PyMuPDF** (`pip install pymupdf`).


In [28]:
# Install PyMuPDF if missing (uncomment if needed)
# %pip install --quiet pymupdf

import fitz  # PyMuPDF
import re
import shutil
from pathlib import Path
from collections import defaultdict

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
REPO_ROOT = Path("/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026")
RESULTS_DIR = REPO_ROOT / "Results"
OUTPUT_DIR_TEMPLATE = "_iteration_{n}_"

PAGE_W = 595.276
PAGE_H_MAX = 841.890
MARGIN = 36
GUTTER_X = 14
ROW_GAP = 18
CAP_GAP = 6
CAP_SIZE = 9

BODY_FONT = "helvetica"

PLOT_PREFIXES = [
    ("confusion_matrix_", "Confusion Matrix"),
    ("metrics_bar_",      "Per-Class Metrics"),
    ("precision_recall_", "Precision-Recall Curve"),
    ("pr_curve_",         "Precision-Recall Curve"),
    ("roc_curve_",        "ROC Curve"),
]

ML_SUFFIXES = [
    "_Random_Forest_Optimized", "_LightGBM_Optimized", "_XGBoost_Optimized",
    "_Voting_Ensemble", "_Random_Forest", "_LightGBM", "_XGBoost", "_SVM",
    "_random_forest", "_svm_linear", "_svm_rbf", "_lightgbm", "_xgboost",
]

SKIP_PATTERNS = (
    "comparison_", "best_macro_f1", "precision_recall_scatter",
    "aggregate_",
)

LANGUAGE_ALIASES = {
    "netherlands": "dutch",
    "germany":     "german",
    "sweden":      "swedish",
}

PLOT_ORDER = [
    "Confusion Matrix",
    "Per-Class Metrics",
    "Precision-Recall Curve",
    "ROC Curve",
]

# Iterations rendered as a single "combine everything" grid PDF.
COMBINE_ALL_ITERS: set = set()

# Themed splits for iter 0 (EDA + language-detection confusion matrices).
ITER0_THEMES = [
    ("iter0_language_detection_confusion_matrices", [
        "confusion_matrix_manual",
        "confusion_matrix_langdetect",
        "confusion_matrix_lingua",
    ]),
    ("iter0_case_counts_by_country_and_type", [
        "Number_of_Cases_by_Country",
        "Number_of_Cases_by_Case_Type",
    ]),
    ("iter0_character_and_word_count_distributions", [
        "Character_Count_Distribution",
        "Word_Count_Distribution",
    ]),
    ("iter0_word_frequency_and_histograms", [
        "Word_Count_Frequency_LogScale",
        "Word_Count_Histogram_Title",
        "Word_Count_Histogram_Description",
    ]),
]

# Themed splits for iter 9.
ITER9_THEMES = [
    ("iter9_test_confusion_matrices", [
        "confusion_matrix_test",
        "confusion_matrix_test_normalized",
    ]),
    ("iter9_per_class_metrics", [
        "per_class_f1",
        "top_misclassifications",
        "precision_vs_recall",
        "support_vs_f1",
    ]),
    ("iter9_calibration_and_country_distributions", [
        "confidence_calibration",
        "confidence_histogram",
        "country_hazard_stacked",
        "country_macro_f1",
        "hazard_distribution_gold_vs_pred",
    ]),
]

# Iterations rendered as one combined PDF of all `confusion_matrix_*` files.
COMBINE_CM_ITERS = {1}

# Iterations rendered as one combined PDF per language.
COMBINE_BY_LANG_ITERS = {2, 3}


# ---------------------------------------------------------------------------
# Filename parsing & grouping
# ---------------------------------------------------------------------------
def _strip_ml_suffix(rest: str):
    for suf in ML_SUFFIXES:
        if rest.endswith(suf):
            return rest[: -len(suf)], suf.lstrip("_")
    return rest, None


def parse_filename(stem: str):
    plot_label = None
    rest = None
    matched_prefix = None
    for prefix, label in PLOT_PREFIXES:
        if stem.startswith(prefix):
            plot_label = label
            rest = stem[len(prefix):]
            matched_prefix = prefix
            break
    if rest is None:
        return None

    if "_manual_" in rest:
        rest_no_ml, ml_model = _strip_ml_suffix(rest)
        language, embedding = rest_no_ml.split("_manual_", 1)
        language = LANGUAGE_ALIASES.get(language.lower(), language)
        return plot_label, language, embedding, ml_model

    if matched_prefix == "confusion_matrix_":
        if rest.endswith("_raw"):
            return None
        if rest.endswith("_normalised") or rest.endswith("_normalized"):
            rest = rest.rsplit("_", 1)[0]
    language = LANGUAGE_ALIASES.get(rest.lower(), rest)
    return plot_label, language, None, None


def safe_name(s: str) -> str:
    return re.sub(r"[^\w\-.]+", "_", s).strip("_")


def _humanize(stem: str) -> str:
    return stem.replace("_", " ").strip()


def iter_pdfs(iter_dir: Path):
    yield from sorted(iter_dir.glob("*.pdf"))
    figures = iter_dir / "figures"
    if figures.is_dir():
        yield from sorted(figures.glob("*.pdf"))


def collect_groups(iter_dir: Path):
    groups: dict = defaultdict(dict)
    skipped = 0
    for pdf in iter_pdfs(iter_dir):
        if any(pdf.name.startswith(p) for p in SKIP_PATTERNS):
            skipped += 1
            continue
        parsed = parse_filename(pdf.stem)
        if parsed is None:
            skipped += 1
            continue
        plot_label, lang, embed, ml = parsed
        groups[(lang, embed, ml)].setdefault(plot_label, pdf)
    return groups, skipped


def discover_iterations() -> list:
    pat = re.compile(r"^_iteration_(\d+)$")
    found = []
    for p in RESULTS_DIR.iterdir():
        if not p.is_dir():
            continue
        m = pat.match(p.name)
        if m:
            found.append((int(m.group(1)), p))
    found.sort(key=lambda x: x[0])
    return found


# ---------------------------------------------------------------------------
# Layout & rendering
# ---------------------------------------------------------------------------
SUB_LETTERS = [f"({chr(ord('a') + i)})" for i in range(26)]


def _grid_cols(n: int) -> int:
    return 1 if n <= 1 else 2


def _natural_size(src_path: Path):
    with fitz.open(src_path) as src:
        r = src[0].rect
        return r.width, r.height


def _render_grid(out_path: Path, items: list, allow_overflow: bool) -> bool:
    if not items:
        return False

    cols = _grid_cols(len(items))
    rows = (len(items) + cols - 1) // cols
    cell_w = (PAGE_W - 2 * MARGIN - GUTTER_X * (cols - 1)) / cols

    rendered = []
    for label, src_path in items:
        sw, sh = _natural_size(src_path)
        scale = min(cell_w / sw, 1.0)
        rendered.append({"label": label, "src": src_path,
                         "w": sw * scale, "h": sh * scale})

    cap_total = CAP_GAP + CAP_SIZE
    row_heights = [
        max(it["h"] for it in rendered[r * cols:(r + 1) * cols]) + cap_total
        for r in range(rows)
    ]
    content_h = sum(row_heights) + ROW_GAP * (rows - 1)
    page_h = content_h + 2 * MARGIN

    if not allow_overflow and page_h > PAGE_H_MAX:
        avail = PAGE_H_MAX - 2 * MARGIN - ROW_GAP * (rows - 1) - cap_total * rows
        natural_row_max = sum(
            max(it["h"] for it in rendered[r * cols:(r + 1) * cols])
            for r in range(rows)
        )
        shrink = avail / natural_row_max
        for it in rendered:
            it["w"] *= shrink
            it["h"] *= shrink
        row_heights = [
            max(it["h"] for it in rendered[r * cols:(r + 1) * cols]) + cap_total
            for r in range(rows)
        ]
        page_h = PAGE_H_MAX

    doc = fitz.open()
    try:
        page = doc.new_page(width=PAGE_W, height=page_h)
        y_cursor = MARGIN
        for r in range(rows):
            row_items = rendered[r * cols:(r + 1) * cols]
            for c, it in enumerate(row_items):
                idx = r * cols + c
                col_x0 = MARGIN + c * (cell_w + GUTTER_X)
                px = col_x0 + (cell_w - it["w"]) / 2
                py = y_cursor
                rect = fitz.Rect(px, py, px + it["w"], py + it["h"])
                with fitz.open(it["src"]) as src:
                    page.show_pdf_page(rect, src, 0)

                letter = SUB_LETTERS[idx] if idx < len(SUB_LETTERS) else f"({idx + 1})"
                sub = f"{letter} {it['label']}."
                tw = fitz.get_text_length(sub, fontname=BODY_FONT, fontsize=CAP_SIZE)
                cap_x = col_x0 + (cell_w - tw) / 2
                cap_y = py + it["h"] + CAP_GAP + CAP_SIZE - 2
                page.insert_text((cap_x, cap_y), sub,
                                 fontname=BODY_FONT, fontsize=CAP_SIZE)
            y_cursor += row_heights[r] + ROW_GAP

        doc.save(out_path, deflate=True, garbage=4)
        return True
    finally:
        doc.close()


def render_group_pdf(out_path: Path, plots: dict) -> bool:
    items = []
    for label in PLOT_ORDER:
        if label in plots:
            items.append((label, plots[label]))
    return _render_grid(out_path, items, allow_overflow=False)


def render_combine_all_pdf(out_path: Path, iter_dir: Path) -> int:
    pdfs = list(iter_pdfs(iter_dir))
    if not pdfs:
        return 0
    items = [(_humanize(p.stem), p) for p in pdfs]
    return 1 if _render_grid(out_path, items, allow_overflow=True) else 0


def _render_themed(out_dir: Path, iter_dir: Path, themes: list,
                   misc_stem: str) -> int:
    figures = iter_dir / "figures"
    by_stem = {p.stem: p for p in iter_dir.glob("*.pdf")}
    if figures.is_dir():
        for p in figures.glob("*.pdf"):
            by_stem.setdefault(p.stem, p)

    written = 0
    used: set = set()
    for stem_out, stems in themes:
        items = []
        for s in stems:
            if s in by_stem:
                items.append((_humanize(s), by_stem[s]))
                used.add(s)
        if not items:
            continue
        out_path = out_dir / f"{stem_out}.pdf"
        if _render_grid(out_path, items, allow_overflow=True):
            written += 1

    leftovers = [(s, by_stem[s]) for s in sorted(by_stem) if s not in used]
    if leftovers:
        items = [(_humanize(s), p) for s, p in leftovers]
        if _render_grid(out_dir / f"{misc_stem}.pdf", items,
                        allow_overflow=True):
            written += 1
    return written


def render_iter9_themed(out_dir: Path, iter_dir: Path) -> int:
    return _render_themed(out_dir, iter_dir, ITER9_THEMES, "iter9_other")


def render_iter0_themed(out_dir: Path, iter_dir: Path) -> int:
    return _render_themed(out_dir, iter_dir, ITER0_THEMES, "iter0_other")


def render_combine_cm_pdf(out_path: Path, iter_dir: Path) -> int:
    pdfs = sorted(iter_dir.glob("confusion_matrix_*.pdf"))
    if not pdfs:
        return 0
    items = []
    for p in pdfs:
        suffix = p.stem[len("confusion_matrix_"):]
        items.append((_humanize(suffix), p))
    return 1 if _render_grid(out_path, items, allow_overflow=True) else 0


def render_combine_by_language(out_dir: Path, iter_dir: Path, iter_num: int) -> int:
    by_lang: dict = defaultdict(list)
    for pdf in iter_pdfs(iter_dir):
        if any(pdf.name.startswith(p) for p in SKIP_PATTERNS):
            continue
        parsed = parse_filename(pdf.stem)
        if parsed is None:
            continue
        plot_label, lang, embed, _ml = parsed
        if plot_label != "Confusion Matrix" or embed is None:
            continue
        by_lang[lang].append((embed, pdf))

    written = 0
    for lang, pairs in sorted(by_lang.items()):
        pairs.sort(key=lambda x: x[0].lower())
        items = [(embed, pdf) for embed, pdf in pairs]
        out_path = out_dir / f"iter{iter_num}_confusion_matrices_{safe_name(lang)}.pdf"
        if _render_grid(out_path, items, allow_overflow=True):
            written += 1
    return written


# ---------------------------------------------------------------------------
# Filename builder for grouped output
# ---------------------------------------------------------------------------
def output_filename(lang, embed, ml) -> str:
    parts = [safe_name(lang)]
    if embed is not None:
        parts.append(safe_name(embed))
    if ml is not None:
        parts.append(safe_name(ml))
    return "__".join(parts) + ".pdf"


# ---------------------------------------------------------------------------
# Driver
# ---------------------------------------------------------------------------
def main():
    iters = discover_iterations()
    if not iters:
        print("No `_iteration_<N>` folders under Results/.")
        return

    total_written = 0
    for iter_num, iter_dir in iters:
        out_dir = RESULTS_DIR / OUTPUT_DIR_TEMPLATE.format(n=iter_num)
        if out_dir.exists():
            shutil.rmtree(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n=== iteration {iter_num} ===")
        print(f"  source : {iter_dir}")
        print(f"  output : {out_dir}")

        written = 0
        if iter_num == 0:
            written = render_iter0_themed(out_dir, iter_dir)
            print(f"  mode   : iter0-themed")
            print(f"  wrote  : {written} PDFs")
        elif iter_num == 9:
            written = render_iter9_themed(out_dir, iter_dir)
            print(f"  mode   : iter9-themed")
            print(f"  wrote  : {written} PDFs")
        elif iter_num in COMBINE_ALL_ITERS:
            out_path = out_dir / f"iter{iter_num}_combined.pdf"
            written = render_combine_all_pdf(out_path, iter_dir)
            print(f"  mode   : combine-all")
            print(f"  wrote  : {written} PDF ({out_path.name})")
        elif iter_num in COMBINE_CM_ITERS:
            out_path = out_dir / f"iter{iter_num}_bert_base_uncased_confusion_matrices.pdf"
            written = render_combine_cm_pdf(out_path, iter_dir)
            print(f"  mode   : combine-confusion-matrices")
            print(f"  wrote  : {written} PDF ({out_path.name})")
        elif iter_num in COMBINE_BY_LANG_ITERS:
            written = render_combine_by_language(out_dir, iter_dir, iter_num)
            print(f"  mode   : combine-by-language")
            print(f"  wrote  : {written} PDFs (one per language)")
        else:
            groups, skipped = collect_groups(iter_dir)
            print(f"  groups : {len(groups)}")
            print(f"  skipped: {skipped} (aggregate / unparseable / non-pdf)")
            prefix = f"iter{iter_num}_" if iter_num in {7, 8} else ""
            for (lang, embed, ml), plots in sorted(
                groups.items(),
                key=lambda kv: tuple("" if x is None else str(x) for x in kv[0]),
            ):
                if not plots:
                    continue
                out_path = out_dir / (prefix + output_filename(lang, embed, ml))
                if render_group_pdf(out_path, plots):
                    written += 1
            print(f"  wrote  : {written} combined PDFs")

        total_written += written

    print(f"\nDone. {total_written} PDFs across {len(iters)} iterations.")


main()



=== iteration 0 ===
  source : /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_0
  output : /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_0_
  mode   : iter0-themed
  wrote  : 4 PDFs

=== iteration 1 ===
  source : /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_1
  output : /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_1_
  mode   : combine-confusion-matrices
  wrote  : 1 PDF (iter1_bert_base_uncased_confusion_matrices.pdf)

=== iteration 2 ===
  source : /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_2
  output : /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_2_
  mode   : combine-by-language
  wrote  : 4 PDFs (one per language)

=== iteration 3 ===
  source : /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_3
  output : /Users/shariarimrozekhan/Documents/GitHub/masterThe